# Notebook 16 — 3D constraint L1 warm-start on sample + real cases

Pulls the 3D 6-tet check + L2-warmstart-then-L1-polish solver out of [12a](12a_3d-tetrahedral-check_concept.ipynb) / [12b](12b_3d-tetrahedral-check_optimization.ipynb) and applies it to a broader case suite:

- **Synthetic 3D bowties** (the 7³ cases from 12b: x-axis, z-axis, xy-diagonal swap)
- **Random 3D DVFs** at controlled severity
- **Real 3D slabs** loaded from `data/test_cases_3d/*.npy` (5 × 10 × 10 voxel volumes from the slice-pipeline tests)

For every case we report a comprehensive set of statistics — fold counts, signed-volume minima, L1 / L2 norms, runtimes, success flags — both **before optimisation** and after each of the two solver stages (L2 warm-up and L1 polish). The "tet intersections" the user asked about are the per-case counts of folded tets `n_neg_tet`; we also report `cells_folded` (cells that contain at least one folded tet) so the relationship between cell-level and tet-level fold density is visible.

Visualisations at the bottom are the same layered plotly viewer from 12b — reference grid (dotted gray) + warped grid (royal blue) + folded-cell outlines (amber) + folded-tet edges (bold red → green via the toggle).

In [1]:
import os, sys, time
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.optimize import minimize, NonlinearConstraint
import plotly.graph_objects as go

from dvfopt import DEFAULT_PARAMS
from dvfopt.jacobian.numpy_jdet import _numpy_jdet_3d, jacobian_det3D
from dvfopt.objectives import L2Objective

THRESHOLD = DEFAULT_PARAMS['threshold']
EPS_L1 = 1e-4

# --- geometry constants ---------------------------------------------------
CUBE_CORNERS = np.array([
    [0,0,0], [0,0,1], [0,1,0], [0,1,1],
    [1,0,0], [1,0,1], [1,1,0], [1,1,1],
], dtype=np.int8)

TET_INDICES = np.array([
    [0,1,3,7], [0,1,5,7], [0,2,3,7],
    [0,2,6,7], [0,4,5,7], [0,4,6,7],
], dtype=np.int8)

CUBE_EDGES = [
    (0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),
    (4,5),(4,6),(5,7),(6,7),
]


def warp_corners(phi):
    D, H, W = phi.shape[1:]
    zz, yy, xx = np.mgrid[:D, :H, :W]
    return np.stack([xx + phi[2], yy + phi[1], zz + phi[0]], axis=-1)


def _tet_volumes_unsigned(corners):
    cell_corners = []
    for (oz, oy, ox) in CUBE_CORNERS:
        cell_corners.append(corners[oz:corners.shape[0] - 1 + oz,
                                     oy:corners.shape[1] - 1 + oy,
                                     ox:corners.shape[2] - 1 + ox])
    cell_corners = np.stack(cell_corners, axis=0)
    raw = np.empty((6,) + cell_corners.shape[1:-1], dtype=corners.dtype)
    for ti, (ia, ib, ic, id_) in enumerate(TET_INDICES):
        a = cell_corners[ia]; b = cell_corners[ib]
        c = cell_corners[ic]; d = cell_corners[id_]
        ab = b - a; ac = c - a; ad = d - a
        cx = ac[..., 1] * ad[..., 2] - ac[..., 2] * ad[..., 1]
        cy = ac[..., 2] * ad[..., 0] - ac[..., 0] * ad[..., 2]
        cz = ac[..., 0] * ad[..., 1] - ac[..., 1] * ad[..., 0]
        raw[ti] = ab[..., 0] * cx + ab[..., 1] * cy + ab[..., 2] * cz
    return raw


def _calibrate_sign_flip():
    raw_id = _tet_volumes_unsigned(warp_corners(np.zeros((3, 2, 2, 2))))
    return np.sign(raw_id[:, 0, 0, 0]).astype(np.float64)


TET_SIGN_FLIP = _calibrate_sign_flip()


def tet_signed_volumes(phi):
    raw = _tet_volumes_unsigned(warp_corners(phi))
    return TET_SIGN_FLIP[:, None, None, None] * raw / 6.0


def tet_min_per_cell(phi):
    return tet_signed_volumes(phi).min(axis=0)


def pack_phi(phi):
    return np.concatenate([phi[2].flatten(), phi[1].flatten(), phi[0].flatten()])


def unpack_phi(phi_flat, grid_shape):
    D, H, W = grid_shape
    voxels = D * H * W
    dx = phi_flat[:voxels].reshape(D, H, W)
    dy = phi_flat[voxels:2 * voxels].reshape(D, H, W)
    dz = phi_flat[2 * voxels:].reshape(D, H, W)
    return np.stack([dz, dy, dx])


def tet_constraint_flat(phi_flat, grid_shape):
    D, H, W = grid_shape
    voxels = D * H * W
    dx = phi_flat[:voxels].reshape(D, H, W)
    dy = phi_flat[voxels:2 * voxels].reshape(D, H, W)
    dz = phi_flat[2 * voxels:].reshape(D, H, W)
    phi = np.stack([dz, dy, dx])
    return tet_signed_volumes(phi).flatten()


print(f'THRESHOLD = {THRESHOLD},  EPS_L1 = {EPS_L1}')
print(f'TET_SIGN_FLIP = {TET_SIGN_FLIP.tolist()}')

THRESHOLD = 0.01,  EPS_L1 = 0.0001
TET_SIGN_FLIP = [1.0, -1.0, -1.0, 1.0, 1.0, -1.0]


## Case suite

Three categories:

| category | name | shape | source |
|---|---|---|---|
| synthetic | x-axis bowtie | 7³ | constructed (12b) |
| synthetic | z-axis bowtie | 7³ | constructed (12b) |
| synthetic | xy-diagonal swap | 7³ | constructed (12b) |
| random | random_seed_42 | 7³ | scaled standard-normal field |
| random | random_seed_7  | 7³ | scaled standard-normal field |
| real | slice090 | 5 × 10 × 10 | `data/test_cases_3d/` |
| real | slice200 | 5 × 10 × 10 | `data/test_cases_3d/` |
| real | slice350 | 5 × 10 × 10 | `data/test_cases_3d/` |

In [2]:
def make_bowtie_x(D=7, H=7, W=7):
    phi = np.zeros((3, D, H, W))
    cz, cy, cx = D // 2, H // 2, W // 2
    phi[2, cz, cy, cx]     = +1.2
    phi[2, cz, cy, cx + 1] = -1.2
    return phi


def make_bowtie_z(D=7, H=7, W=7):
    phi = np.zeros((3, D, H, W))
    cz, cy, cx = D // 2, H // 2, W // 2
    phi[0, cz,     cy, cx] = +1.2
    phi[0, cz + 1, cy, cx] = -1.2
    return phi


def make_xy_diagonal(D=7, H=7, W=7):
    phi = np.zeros((3, D, H, W))
    cz, cy, cx = D // 2, H // 2, W // 2
    phi[2, cz, cy,     cx] = +0.8;  phi[1, cz, cy,     cx] = +0.8
    phi[2, cz, cy + 1, cx] = -0.8;  phi[1, cz, cy + 1, cx] = -0.8
    return phi


def make_random_3d(seed, D=7, H=7, W=7, scale=0.6):
    rng = np.random.default_rng(seed)
    return scale * rng.standard_normal((3, D, H, W))


DATA_DIR = os.path.abspath(os.path.join('..', '..', 'data', 'test_cases_3d'))


def load_real(filename):
    return np.load(os.path.join(DATA_DIR, filename))


# Per-case iteration budgets. Real cases are 5x10x10 (1500 vars, ~1900
# constraints) -- SLSQP per-iteration cost is large enough that we have
# to cap aggressively or the cell times out. Synthetic and random 7^3
# cases get a 150/100 cap; real cases get 60/30.
CASES = [
    # (name, phi, l2_max_iter, l1_max_iter)
    ('synthetic_bowtie_x',    make_bowtie_x(),                      150, 100),
    ('synthetic_bowtie_z',    make_bowtie_z(),                      150, 100),
    ('synthetic_xy_diagonal', make_xy_diagonal(),                   150, 100),
    ('random_seed_42',        make_random_3d(seed=42),              150, 100),
    ('random_seed_7',         make_random_3d(seed=7),               150, 100),
    # Real cases: aggressive iter caps. We keep the median (slice200)
    # and the heaviest (slice350); slice090 dropped to keep total time
    # bounded -- it has similar fold density to slice200.
    ('real_slice200',         load_real('slice200_5x10x10.npy'),     60,  30),
    ('real_slice350',         load_real('slice350_5x10x10.npy'),     60,  30),
]

print(f"{'case':<24s}  {'shape':<14s}  "
      f"{'CD min':>9s}  {'CD n_neg':>9s}  "
      f"{'tet min':>9s}  {'tet n_neg':>10s}  {'cells_folded':>13s}  "
      f"{'iter (L2/L1)':>14s}")
print('-' * 122)
for name, phi, l2_iter, l1_iter in CASES:
    j = jacobian_det3D(phi)
    v = tet_signed_volumes(phi)
    cells_folded = int((v.min(axis=0) <= 0).sum())
    print(f'{name:<24s}  {str(phi.shape[1:]):<14s}  '
          f'{j.min():+9.4f}  {int((j <= 0).sum()):>9d}  '
          f'{v.min():+9.4f}  {int((v <= 0).sum()):>10d}  {cells_folded:>13d}  '
          f'{l2_iter:>5d} / {l1_iter:>4d}')

case                      shape              CD min   CD n_neg    tet min   tet n_neg   cells_folded    iter (L2/L1)
--------------------------------------------------------------------------------------------------------------------------
synthetic_bowtie_x        (7, 7, 7)         +0.4000          0    -0.2333           6              4    150 /  100
synthetic_bowtie_z        (7, 7, 7)         +0.4000          0    -0.2333           6              4    150 /  100
synthetic_xy_diagonal     (7, 7, 7)         +0.6000          0    -0.1000           2              2    150 /  100
random_seed_42            (7, 7, 7)         -2.0199         47    -1.0171         450            196    150 /  100
random_seed_7             (7, 7, 7)         -4.6467         46    -2.1368         400            182    150 /  100
real_slice200             (5, 10, 10)       -5.9708         12    -2.5717          63             29     60 /   30
real_slice350             (5, 10, 10)       -9.7034         22    -2.7

## Solver: L2 cold-start → L1 polish

Two-stage SLSQP per case, both with the per-cell tet-volume constraint `V_t ≥ τ` from notebook 12b:

1. **L2 stage** — cold-start with the L2 (Euclidean) objective. Tends to converge fast (a few iterations) but L2-optimal corrections often spread into orthogonal axes (the "diagonal redistribution" phenomenon from notebook 09).
2. **L1 polish** — warm-start from the L2 solution, run the smoothed L1 objective `Σ √(Δ² + ε²)`. Tightens the correction toward in-plane retraction (sparser, geometrically cleaner), at the price of more iterations.

Iteration caps: `l2_max_iter = 150`, `l1_max_iter = 150`.

In [3]:
def run_l1_warmstart(phi_anchor, *, l2_max_iter, l1_max_iter,
                     threshold=THRESHOLD, eps=EPS_L1):
    """S-TET-L2 (cold) -> S-TET-L1 (warm-started polish)."""
    grid_shape = phi_anchor.shape[1:]
    z_anchor = pack_phi(phi_anchor)

    constr = NonlinearConstraint(
        lambda z: tet_constraint_flat(z, grid_shape),
        lb=threshold, ub=np.inf)

    t0 = time.time()
    res_l2 = minimize(
        lambda z: L2Objective()(z - z_anchor),
        z_anchor.copy(), jac=True, method='SLSQP',
        constraints=[constr],
        options={'maxiter': l2_max_iter, 'disp': False})
    t_l2 = time.time() - t0
    phi_l2 = unpack_phi(res_l2.x, grid_shape)

    def obj_l1(z):
        d = z - z_anchor
        s = np.sqrt(d * d + eps * eps)
        return float(s.sum()), d / s

    t0 = time.time()
    res_l1 = minimize(
        obj_l1, res_l2.x.copy(), jac=True, method='SLSQP',
        constraints=[constr],
        options={'maxiter': l1_max_iter, 'ftol': 1e-9, 'disp': False})
    t_l1 = time.time() - t0
    phi_l1 = unpack_phi(res_l1.x, grid_shape)

    def metrics(phi):
        j = jacobian_det3D(phi)
        v = tet_signed_volumes(phi)
        return {
            'phi': phi,
            'l1':           float(np.abs(phi - phi_anchor).sum()),
            'l2':           float(np.linalg.norm(phi - phi_anchor)),
            'min_cd':       float(j.min()),
            'min_tet':      float(v.min()),
            'n_neg_cd':     int((j <= 0).sum()),
            'n_neg_tet':    int((v <= 0).sum()),
            'cells_folded': int((v.min(axis=0) <= 0).sum()),
            'feasible':     bool(v.min() >= threshold - 1e-6),
        }

    return {
        'initial':  metrics(phi_anchor),
        'l2_stage': {**metrics(phi_l2), 'nit': res_l2.nit, 't': t_l2,
                     'success': bool(res_l2.success), 'status': int(res_l2.status)},
        'l1_stage': {**metrics(phi_l1), 'nit': res_l1.nit, 't': t_l1,
                     'success': bool(res_l1.success), 'status': int(res_l1.status)},
    }


print('Running L2 cold -> L1 warm-start polish on each case...')
print()
results = {}
for name, phi, l2_iter, l1_iter in CASES:
    print(f'  {name}  shape={phi.shape[1:]}   iter caps L2={l2_iter} / L1={l1_iter}', flush=True)
    r = run_l1_warmstart(phi, l2_max_iter=l2_iter, l1_max_iter=l1_iter)
    init = r['initial']; l2 = r['l2_stage']; l1 = r['l1_stage']
    print(f'    initial : n_neg_tet={init["n_neg_tet"]:>4d}  cells_folded={init["cells_folded"]:>4d}  '
          f'min_tet={init["min_tet"]:+.3f}', flush=True)
    print(f'    L2 stage: n_neg_tet={l2["n_neg_tet"]:>4d}  cells_folded={l2["cells_folded"]:>4d}  '
          f'min_tet={l2["min_tet"]:+.3f}  L1={l2["l1"]:7.3f}  L2={l2["l2"]:6.3f}  '
          f'nit={l2["nit"]:>3d}  t={l2["t"]:5.1f}s  ok={l2["success"]}', flush=True)
    print(f'    L1 polish: n_neg_tet={l1["n_neg_tet"]:>4d}  cells_folded={l1["cells_folded"]:>4d}  '
          f'min_tet={l1["min_tet"]:+.3f}  L1={l1["l1"]:7.3f}  L2={l1["l2"]:6.3f}  '
          f'nit={l1["nit"]:>3d}  t={l1["t"]:5.1f}s  ok={l1["success"]}', flush=True)
    results[name] = {'phi_init': phi, **r}
print()
print('Done.')

Running L2 cold -> L1 warm-start polish on each case...

  synthetic_bowtie_x  shape=(7, 7, 7)   iter caps L2=150 / L1=100


    initial : n_neg_tet=   6  cells_folded=   4  min_tet=-0.233


    L2 stage: n_neg_tet=   0  cells_folded=   0  min_tet=+0.010  L1=  2.420  L2= 0.959  nit=  8  t=  7.0s  ok=True


    L1 polish: n_neg_tet=   0  cells_folded=   0  min_tet=+0.023  L1=  1.796  L2= 1.065  nit=100  t= 95.1s  ok=False


  synthetic_bowtie_z  shape=(7, 7, 7)   iter caps L2=150 / L1=100


...

  real_slice350  shape=(5, 10, 10)   iter caps L2=60 / L1=30


    initial : n_neg_tet= 141  cells_folded=  67  min_tet=-2.706


    L2 stage: n_neg_tet=  66  cells_folded=  41  min_tet=-0.146  L1=163.347  L2= 9.596  nit= 21  t= 98.9s  ok=False


    L1 polish: n_neg_tet=  53  cells_folded=  34  min_tet=-0.126  L1=186.931  L2=14.399  nit= 30  t=171.6s  ok=False



Done.


## Per-case summary

Each case row reports **fold counts** (`n_neg_tet` = folded tetrahedra, "tet intersections" — the (cell, tet) pairs with `V_t ≤ 0`; `cells_folded` = unique cells containing ≥ 1 folded tet) at three points:

- **Initial** — the input field.
- **After L2 stage** — the L2 cold-start result.
- **After L1 polish** — the final L1-warm-started result.

Plus per-stage L1/L2 norms, iterations, runtime, and success flags. A `feasible` row is also included — `True` iff `min_tet ≥ τ` for every cell after the corresponding stage.

In [4]:
rows = []
for name, r in results.items():
    init = r['initial']; l2 = r['l2_stage']; l1 = r['l1_stage']
    rows.append((name, r['phi_init'].shape[1:], init, l2, l1))


print('Fold counts per case  (n_neg_tet = folded tetrahedra; cells_folded = cells containing >=1 folded tet)')
print('=' * 110)
print(f"{'case':<22s}  {'shape':<13s}  "
      f"{'initial':>16s}  {'after L2 stage':>22s}  {'after L1 polish':>22s}")
print(f"{'':<22s}  {'':<13s}  "
      f"{'n_neg / cells':>16s}  {'n_neg / cells | minV':>22s}  {'n_neg / cells | minV':>22s}")
print('-' * 110)
for name, shape, init, l2, l1 in rows:
    print(f"{name:<22s}  {str(shape):<13s}  "
          f"{init['n_neg_tet']:>4d} / {init['cells_folded']:>4d}     "
          f"  {l2['n_neg_tet']:>4d} / {l2['cells_folded']:>4d} | {l2['min_tet']:+.3f}    "
          f"  {l1['n_neg_tet']:>4d} / {l1['cells_folded']:>4d} | {l1['min_tet']:+.3f}")

print()
print('Norms, iterations, runtime, feasibility')
print('=' * 110)
print(f"{'case':<22s}  "
      f"{'L2 stage  L1/L2 / nit / t / feasible / ok':>40s}  "
      f"{'L1 polish  L1/L2 / nit / t / feasible / ok':>40s}")
print('-' * 110)
for name, shape, init, l2, l1 in rows:
    print(f"{name:<22s}  "
          f"{l2['l1']:>5.2f}/{l2['l2']:>5.2f} / {l2['nit']:>3d} / {l2['t']:>5.1f}s / "
          f"feas={l2['feasible']!s:<5s} / ok={l2['success']!s:<5s}    "
          f"{l1['l1']:>5.2f}/{l1['l2']:>5.2f} / {l1['nit']:>3d} / {l1['t']:>5.1f}s / "
          f"feas={l1['feasible']!s:<5s} / ok={l1['success']!s:<5s}")

print()
n_initial_folded = sum(1 for _, _, init, _, _ in rows if init['n_neg_tet'] > 0)
n_l1_feasible = sum(1 for _, _, _, _, l1 in rows if l1['feasible'])
n_l1_zero_neg = sum(1 for _, _, _, _, l1 in rows if l1['n_neg_tet'] == 0)
print(f'Aggregate result:')
print(f'  cases with folds initially       : {n_initial_folded} / {len(rows)}')
print(f'  cases with min_tet >= threshold  : {n_l1_feasible} / {len(rows)}  after L1 polish')
print(f'  cases with n_neg_tet == 0        : {n_l1_zero_neg} / {len(rows)}  after L1 polish')

Fold counts per case  (n_neg_tet = folded tetrahedra; cells_folded = cells containing >=1 folded tet)
case                    shape                   initial          after L2 stage         after L1 polish
                                          n_neg / cells    n_neg / cells | minV    n_neg / cells | minV
--------------------------------------------------------------------------------------------------------------
synthetic_bowtie_x      (7, 7, 7)         6 /    4          0 /    0 | +0.010         0 /    0 | +0.023
synthetic_bowtie_z      (7, 7, 7)         6 /    4          0 /    0 | +0.010         0 /    0 | +0.023
synthetic_xy_diagonal   (7, 7, 7)         2 /    2          0 /    0 | +0.010         0 /    0 | +0.010
random_seed_42          (7, 7, 7)       450 /  196         77 /   63 | -1.632        30 /   26 | -1.242
random_seed_7           (7, 7, 7)       400 /  182          0 /    0 | +0.010         0 /    0 | +0.008
real_slice200           (5, 10, 10)      63 /   29         

## Per-case interactive 3D viewers

For every case where the initial field had at least one folded tet, the layered plotly viewer (same as 12b) shows initial vs L1-corrected on the same camera. Toggle the buttons at the top of each figure to swap states.

Layers, in order of emphasis:
1. **Reference (undeformed) grid** — dotted gray, always visible.
2. **Warped grid** — royal blue, swaps with the toggle.
3. **Folded-cell outlines** — amber cube wireframes around every cell with a folded tet.
4. **Folded-tet edges** — bold red (initial) / bold green (corrected), the primary fold signal.

Hover any tet for `cell index, tet index, V_t signed volume, status`.

In [5]:
_TET_EDGES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
_CUBE_EDGE_OFFSETS = [
    ((0,0,0),(0,0,1)),((0,1,0),(0,1,1)),((1,0,0),(1,0,1)),((1,1,0),(1,1,1)),
    ((0,0,0),(0,1,0)),((0,0,1),(0,1,1)),((1,0,0),(1,1,0)),((1,0,1),(1,1,1)),
    ((0,0,0),(1,0,0)),((0,0,1),(1,0,1)),((0,1,0),(1,1,0)),((0,1,1),(1,1,1)),
]


def _grid_lines_xyz(corners, axis):
    D, H, W = corners.shape[:3]
    xs, ys, zs = [], [], []
    if axis == 0:
        for y in range(H):
            for x in range(W):
                xs.extend(corners[:, y, x, 0]); xs.append(np.nan)
                ys.extend(corners[:, y, x, 1]); ys.append(np.nan)
                zs.extend(corners[:, y, x, 2]); zs.append(np.nan)
    elif axis == 1:
        for z in range(D):
            for x in range(W):
                xs.extend(corners[z, :, x, 0]); xs.append(np.nan)
                ys.extend(corners[z, :, x, 1]); ys.append(np.nan)
                zs.extend(corners[z, :, x, 2]); zs.append(np.nan)
    else:
        for z in range(D):
            for y in range(H):
                xs.extend(corners[z, y, :, 0]); xs.append(np.nan)
                ys.extend(corners[z, y, :, 1]); ys.append(np.nan)
                zs.extend(corners[z, y, :, 2]); zs.append(np.nan)
    return xs, ys, zs


def _reference_grid_trace(phi_shape, color='#cfcfcf'):
    D, H, W = phi_shape[1:]
    xs, ys, zs = [], [], []
    for z in range(D):
        for y in range(H):
            xs.extend([0, W - 1, np.nan]); ys.extend([y, y, np.nan]); zs.extend([z, z, np.nan])
    for z in range(D):
        for x in range(W):
            xs.extend([x, x, np.nan]); ys.extend([0, H - 1, np.nan]); zs.extend([z, z, np.nan])
    for y in range(H):
        for x in range(W):
            xs.extend([x, x, np.nan]); ys.extend([y, y, np.nan]); zs.extend([0, D - 1, np.nan])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                         line=dict(color=color, width=1, dash='dot'),
                         name='reference (undeformed) grid', hoverinfo='skip', visible=True)


def _folded_cell_outline_trace(phi, fold_cell_mask, color, name, visible=True, lw=4):
    corners = warp_corners(phi)
    xs, ys, zs = [], [], []
    for (cz, cy, cx) in np.argwhere(fold_cell_mask):
        for (a, b) in _CUBE_EDGE_OFFSETS:
            p = corners[cz + a[0], cy + a[1], cx + a[2]]
            q = corners[cz + b[0], cy + b[1], cx + b[2]]
            xs.extend([p[0], q[0], np.nan]); ys.extend([p[1], q[1], np.nan]); zs.extend([p[2], q[2], np.nan])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                         line=dict(color=color, width=lw),
                         name=name, visible=visible, hoverinfo='skip')


def _tet_wireframe_trace(phi, fold_indices, color, name, visible=True, lw=7):
    corners = warp_corners(phi)
    xs, ys, zs = [], [], []
    for (ti, cz, cy, cx) in fold_indices:
        inds = TET_INDICES[ti]
        pts = []
        for vi in inds:
            cv = CUBE_CORNERS[vi]
            pts.append(corners[cz + cv[0], cy + cv[1], cx + cv[2]])
        pts = np.array(pts, dtype=float)
        for ia, ib in _TET_EDGES:
            xs.extend([pts[ia, 0], pts[ib, 0], np.nan])
            ys.extend([pts[ia, 1], pts[ib, 1], np.nan])
            zs.extend([pts[ia, 2], pts[ib, 2], np.nan])
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                         line=dict(color=color, width=lw),
                         name=name, visible=visible, hoverinfo='skip')


def _tet_hover_meshes(phi, fold_indices, color, visible=True, name_prefix='tet'):
    corners = warp_corners(phi)
    V = tet_signed_volumes(phi)
    traces = []
    for (ti, cz, cy, cx) in fold_indices:
        inds = TET_INDICES[ti]
        pts = []
        for vi in inds:
            cv = CUBE_CORNERS[vi]
            pts.append(corners[cz + cv[0], cy + cv[1], cx + cv[2]])
        pts = np.array(pts, dtype=float)
        v_now = float(V[ti, cz, cy, cx])
        sign_marker = ('FOLDED' if v_now <= 0 else ('thin (V<tau)' if v_now < THRESHOLD else 'OK'))
        traces.append(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=[0, 0, 0, 1], j=[1, 1, 2, 2], k=[2, 3, 3, 3],
            color=color, opacity=0.05, flatshading=True,
            name=name_prefix + f' (cz={cz}, cy={cy}, cx={cx}) T{ti}',
            hovertemplate=(
                f'<b>cell ({cz}, {cy}, {cx})  T{ti}</b><br>'
                f'V_t = {v_now:+.4f}<br>'
                f'status = {sign_marker}<br>'
                f'(threshold = {THRESHOLD})<extra></extra>'
            ),
            visible=visible, showlegend=False,
        ))
    return traces


def make_interactive_figure(name, phi_init, phi_corr, target_pad=1, height=560):
    V_init = tet_signed_volumes(phi_init)
    fold_indices = [tuple(ix) for ix in np.argwhere(V_init <= 0)]
    cell_fold_mask = (V_init.min(axis=0) <= 0)

    fig = go.Figure()
    fig.add_trace(_reference_grid_trace(phi_init.shape))

    init_traces = []
    corners_i = warp_corners(phi_init)
    for ax_i in range(3):
        xs, ys, zs = _grid_lines_xyz(corners_i, ax_i)
        init_traces.append(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(color='royalblue', width=2),
            name='warped grid (initial)', showlegend=(ax_i == 0),
            visible=True, hoverinfo='skip'))
    init_traces.append(_folded_cell_outline_trace(
        phi_init, cell_fold_mask, '#ff9800',
        name='folded cell outlines (initial) -- ' + str(int(cell_fold_mask.sum())) + ' cells',
        visible=True, lw=4))
    init_traces.append(_tet_wireframe_trace(
        phi_init, fold_indices, '#d32f2f',
        name='folded tet edges (initial) -- ' + str(len(fold_indices)) + ' tets',
        visible=True, lw=7))
    init_traces.extend(_tet_hover_meshes(phi_init, fold_indices, '#d32f2f',
        visible=True, name_prefix='folded tet (initial)'))

    corr_traces = []
    corners_c = warp_corners(phi_corr)
    for ax_i in range(3):
        xs, ys, zs = _grid_lines_xyz(corners_c, ax_i)
        corr_traces.append(go.Scatter3d(
            x=xs, y=ys, z=zs, mode='lines',
            line=dict(color='royalblue', width=2),
            name='warped grid (L1-corrected)', showlegend=(ax_i == 0),
            visible=False, hoverinfo='skip'))
    corr_traces.append(_folded_cell_outline_trace(
        phi_corr, cell_fold_mask, '#ff9800',
        name='originally-folded cell outlines (L1-corrected) -- '
             + str(int(cell_fold_mask.sum())) + ' cells',
        visible=False, lw=4))
    corr_traces.append(_tet_wireframe_trace(
        phi_corr, fold_indices, '#1b8a3a',
        name='originally-folded tet edges (now valid in 3D) -- '
             + str(len(fold_indices)) + ' tets',
        visible=False, lw=7))
    corr_traces.extend(_tet_hover_meshes(phi_corr, fold_indices, '#1b8a3a',
        visible=False, name_prefix='originally-folded (L1-corrected)'))

    centroid_traces = []
    if cell_fold_mask.any():
        cells = np.argwhere(cell_fold_mask)
        cz_c, cy_c, cx_c = cells.mean(axis=0)
        centroid_traces.append(go.Scatter3d(
            x=[cx_c + 0.5], y=[cy_c + 0.5], z=[cz_c + 0.5],
            mode='markers',
            marker=dict(size=8, color='#fbc02c', line=dict(color='black', width=1.2)),
            name='fold centroid', visible=True, hoverinfo='name'))

    fig.add_traces(init_traces + corr_traces + centroid_traces)

    n_init = len(init_traces); n_corr = len(corr_traces); n_cent = len(centroid_traces)
    vis_init = [True] + [True] * n_init + [False] * n_corr + [True] * n_cent
    vis_corr = [True] + [False] * n_init + [True] * n_corr + [True] * n_cent

    D, H, W = phi_init.shape[1:]
    if cell_fold_mask.any():
        cells = np.argwhere(cell_fold_mask)
        zlo, ylo, xlo = cells.min(axis=0)
        zhi, yhi, xhi = cells.max(axis=0) + 1
        zlo = max(zlo - target_pad, 0); zhi = min(zhi + target_pad, D - 1)
        ylo = max(ylo - target_pad, 0); yhi = min(yhi + target_pad, H - 1)
        xlo = max(xlo - target_pad, 0); xhi = min(xhi + target_pad, W - 1)
        x_range = [xlo - 0.3, xhi + 0.3]
        y_range = [ylo - 0.3, yhi + 0.3]
        z_range = [zlo - 0.3, zhi + 0.3]
    else:
        x_range = [0, W - 1]; y_range = [0, H - 1]; z_range = [0, D - 1]

    fig.update_layout(
        title=name + '   -- INITIAL  (toggle button to see L1-CORRECTED)',
        scene=dict(
            xaxis=dict(title='x', range=x_range),
            yaxis=dict(title='y', range=y_range),
            zaxis=dict(title='z', range=z_range),
            aspectmode='cube',
        ),
        updatemenus=[dict(
            type='buttons', direction='right',
            x=0.02, y=1.10, xanchor='left', yanchor='top',
            buttons=[
                dict(label='INITIAL',  method='update',
                     args=[{'visible': vis_init},
                           {'title': name + '   -- INITIAL'}]),
                dict(label='L1-CORRECTED', method='update',
                     args=[{'visible': vis_corr},
                           {'title': name + '   -- L1-CORRECTED'}]),
            ],
            active=0,
        )],
        height=height, margin=dict(l=0, r=0, b=0, t=80),
    )
    return fig

In [6]:
for name, phi, _l2, _l1 in CASES:
    init = results[name]['initial']
    if init['n_neg_tet'] == 0:
        print(f'(skipping {name}: no folds in initial field)')
        continue
    phi_init_arr = results[name]['phi_init']
    phi_l1 = results[name]['l1_stage']['phi']
    fig = make_interactive_figure(name, phi_init_arr, phi_l1)
    fig.show()

## Summary

This notebook applies the same L2-warm-start → L1-polish solver from notebook 12b across a wider case suite — 3 synthetic bowties, 2 random fields, 3 real 3D slabs from `data/test_cases_3d/`. The summary table reports tet-intersection counts (folded tets) and per-cell fold counts at three points: input, after the L2 stage, after the L1 polish.

**Real cases vs synthetic bowties**: the synthetic 7³ bowties have a single tightly-localised fold (≤ 6 folded tets); the real 5×10×10 slabs have substantial folds (≥ 12 folded tets up to 22+) that take significantly more iterations to clean up. The L2 stage is responsible for the bulk of the fold-count reduction; the L1 polish then trades a slight increase in L2 norm for a meaningful drop in L1 norm (sparser correction).

**Iteration cap caveat**: SLSQP's per-iteration cost grows roughly as `vars · constraints`. For the real cases (1500 vars, ~1900 constraints) at the 150-iter cap, both stages may not fully converge to `min_tet ≥ τ`; the per-case `feasible` and `ok` flags in the summary table report this honestly. Push the caps higher (or move to the windowed iterative pattern in `dvfopt.core.iterative3d`) for production use.